# Modelagem de tópicos: por que comparar com classes é difícil?

Este ensaio simplifica a comparação **LLDA + LDA versus LDA direto**, com K=10 e K=54 nos mesmos 2.000 artigos. Não buscamos um vencedor nem calculamos uma “acurácia de tópicos”.

**Pergunta:** os grupos descobertos espontaneamente correspondem às classes originais de forma simples?

Na modelagem, o modelo descobre grupos e palavras; o número e o significado desses grupos não precisam coincidir com a taxonomia do AAPD. Na classificação, definimos previamente quais classes devem ser atribuídas, tornando a correspondência operacional explícita.

Usamos apenas palavras-chave, exemplos, uma visualização tópico × classe e duas contagens descritivas. Sem sonda supervisionada, alinhamento Húngaro, PCA ou ranking de métricas.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'dataset_loader.py').exists())
EXPERIMENT_DIR = ROOT / 'experimento_aapd_comparativo'
sys.path.insert(0, str(EXPERIMENT_DIR))
from modelagem_simples import run
OUTPUT_DIR = EXPERIMENT_DIR / 'resultados_modelagem_simples'
REEXECUTAR = False  # True refaz os quatro ajustes; False reutiliza os resultados, se disponíveis.

if REEXECUTAR or not (OUTPUT_DIR / 'resumo.csv').exists():
    summary, spread, topics, assignments = run()
else:
    summary = pd.read_csv(OUTPUT_DIR / 'resumo.csv')
    spread = pd.read_csv(OUTPUT_DIR / 'dispersao_por_classe.csv')
    topics = pd.read_csv(OUTPUT_DIR / 'topicos.csv')
    assignments = pd.read_csv(OUTPUT_DIR / 'atribuicoes.csv')
pd.set_option('display.max_colwidth', 110)
display(summary)

,Metodo,K,Documentos,Artigos comuns no diagnostico,Topicos usados,Classes espalhadas em 2+ topicos (80%),Mediana de topicos para 80% da classe,Documentos sem termos
0,LLDA + LDA,10,2000,1986,10,54,6.0,14
1,LLDA + LDA,54,2000,1986,54,54,18.5,14
2,LDA direto,10,2000,1986,10,53,4.0,0
3,LDA direto,54,2000,1986,54,54,19.0,0


## 1. Um protocolo simples e comparável

**LLDA + LDA:** reutilizamos os títulos temáticos do CSV original e ajustamos LDA sobre eles. Os títulos foram gerados com limite de 1.500 caracteres de entrada; a LLM não é chamada novamente.

**LDA direto:** recuperamos o resumo completo, mas usamos seus primeiros 1.500 caracteres para manter o mesmo orçamento de texto da geração original. Isso corrige o uso anterior do campo de 140 caracteres do CSV. A comparação continua sendo entre pipelines: texto original versus uma síntese gerada por LLM.

Ambos usam o mesmo vetorizador configurado com unigramas/bigramas e os mesmos limites de frequência, a mesma semente e 35 iterações, em K=10 e K=54. Os vocabulários são aprendidos separadamente. Quatorze títulos não têm termos no vocabulário após a vetorização. Os diagnósticos usam os mesmos **1.986 artigos com termos nos dois pipelines**, evitando atribuir significado a misturas uniformes desses títulos. O ajuste usa os 2.000 artigos, sem rótulos; é uma descrição exploratória do corpus, não um teste de generalização. O LDA roda em CPU.

As classes só entram depois do ajuste. Cada artigo conta em **todas** as suas classes, sem eleger uma classe primária.

In [2]:
entries = pd.read_csv(OUTPUT_DIR / 'entradas.csv')
display(entries.head())
print('Artigos:', entries.doc_id.nunique())
print('Limite de caracteres do LDA direto:', entries.caracteres_entrada_direta.max())

Artigos: 2000
Limite de caracteres do LDA direto: 1500


,doc_id,fonte_texto,caracteres_resumo_completo,caracteres_entrada_direta,topico_llda
0,51,test.txt:artigo_51,389,389,Graph Edge Rabbit Hunting Problem
1,178,test.txt:artigo_178,1066,1066,Scientific Research Collaboration Dynamics
2,209,test.txt:artigo_209,1588,1500,network seed identification measures
3,228,test.txt:artigo_228,348,348,Quantum code construction from cyclic codes
4,285,test.txt:artigo_285,508,508,Sparse Matrix Storage Format Adaptation


## 2. O que os tópicos significam?

Um tópico é apresentado pelas oito palavras/expressões de maior peso. Para facilitar a leitura, associamos cada artigo ao seu tópico de maior peso e mostramos os grupos mais ocupados. A mistura completa é preservada nos arquivos `theta_*.npz`.

A associação dominante é apenas uma simplificação da visualização. Um artigo pode abordar vários assuntos e ter peso em vários tópicos; isso não significa que ele possua uma única classe.

In [3]:
for (method, k), group in topics.groupby(['Metodo','K'], sort=False):
    print(f'\n{method} | K={k}')
    display(group.nlargest(5, 'Artigos')[['Topico','Artigos','Palavras','Classes frequentes (artigos/total)']])


LLDA + LDA | K=10

LLDA + LDA | K=54

LDA direto | K=10

LDA direto | K=54


,Topico,Artigos,Palavras,Classes frequentes (artigos/total)
6,7,236,"network, social, dynamics, sparse, coding, social network, optimization, network coding",cs.IT: 91/236; math.IT: 91/236; cs.SI: 51/236
7,8,216,"network, optimization, information, design, model, logic, analysis, wireless",cs.IT: 77/216; math.IT: 77/216; cs.LG: 29/216
1,2,211,"channel, capacity, optimization, channel capacity, time, interference, energy, space",cs.IT: 105/211; math.IT: 105/211; cs.LG: 26/211
0,1,207,"problem, control, systems, network, graph, language, complex, natural",cs.IT: 42/207; math.IT: 42/207; cs.CR: 27/207
5,6,197,"algorithm, optimization, artificial, relay, optimization algorithm, code, systems, quantum",cs.IT: 73/197; math.IT: 73/197; cs.AI: 33/197


,Topico,Artigos,Palavras,Classes frequentes (artigos/total)
17,8,81,"optimization, network, network optimization, cognitive, radio, communication, cognitive radio, tree",cs.IT: 39/81; math.IT: 39/81; cs.DS: 11/81
25,16,66,"network, social, social network, modeling, analysis, dynamics, network analysis, model",cs.SI: 27/66; physics.soc-ph: 24/66; cs.IT: 14/66
49,40,63,"algorithm, sparse, optimization, optimization algorithm, representation, matrix, graph, coding",cs.IT: 13/63; math.IT: 13/63; cs.DS: 12/63
12,3,57,"quantum, networks, neural, error, correction, error correction, relaying, mimo",cs.IT: 23/57; math.IT: 23/57; quant-ph: 21/57
41,32,53,"data, security, mining, data mining, knowledge, performance, network security, network",cs.CR: 11/53; cs.IT: 9/53; math.IT: 9/53


,Topico,Artigos,Palavras,Classes frequentes (artigos/total)
66,3,412,"data, learning, based, paper, results, time, network, performance",cs.LG: 137/412; cs.AI: 78/412; stat.ML: 72/412
67,4,284,"algorithm, method, based, proposed, problem, optimization, paper, methods",cs.IT: 124/284; math.IT: 124/284; cs.LG: 71/284
71,8,256,"model, systems, language, information, logic, paper, based, using",cs.LO: 64/256; cs.CL: 51/256; cs.PL: 43/256
69,6,228,"network, power, user, relay, networks, transmission, performance, nodes",cs.IT: 199/228; math.IT: 199/228; cs.NI: 49/228
64,1,209,"algorithm, problem, time, log, bound, number, set, complexity",cs.DS: 69/209; cs.DM: 57/209; cs.CC: 50/209


,Topico,Artigos,Palavras,Classes frequentes (artigos/total)
102,29,131,"channel, power, user, multiple, interference, transmission, optimal, channels",cs.IT: 119/131; math.IT: 119/131; cs.NI: 13/131
124,51,85,"data, social, model, learning, approach, models, user, information",cs.LG: 23/85; cs.SI: 18/85; cs.AI: 15/85
77,4,72,"algorithm, method, optimization, methods, based, problem, proposed, problems",cs.LG: 21/72; cs.IT: 20/72; math.IT: 20/72
87,14,72,"codes, coding, code, decoding, channel, bound, relay, error",cs.IT: 63/72; math.IT: 63/72; math.CO: 8/72
81,8,58,"algorithm, algorithms, problem, based, systems, paper, time, problems",cs.DS: 10/58; cs.LO: 10/58; cs.IT: 8/58


## 3. Uma pergunta fácil de reportar

**Em quantos tópicos precisamos reunir artigos para encontrar 80% dos documentos de uma classe?**

Exemplo: se uma classe tem dez artigos distribuídos em 5, 3 e 2 artigos por tópico dominante, os dois maiores tópicos reúnem oito artigos. O resultado é **dois tópicos para cobrir 80%**.

Reportamos: (a) quantos dos K tópicos são dominantes em algum artigo; (b) quantas das 54 classes precisam de dois ou mais tópicos para reunir 80% de seus artigos. A mediana desse número complementa a leitura.

O limiar de 80% é uma convenção descritiva fixada antes do ensaio, não uma métrica de qualidade. Uma classe espalhada pode conter subtemas legítimos; um tópico pode corresponder a classes que coocorrem. **Dispersão não equivale a erro.**

In [4]:
display(summary[['Metodo','K','Artigos comuns no diagnostico','Topicos usados','Classes espalhadas em 2+ topicos (80%)',
                 'Mediana de topicos para 80% da classe']])
plot = summary.copy()
plot['Configuracao'] = plot.Metodo + ' | K=' + plot.K.astype(str)
px.bar(plot, x='Configuracao', y='Classes espalhadas em 2+ topicos (80%)',
       text='Classes espalhadas em 2+ topicos (80%)', range_y=[0,54],
       title='Quantas das 54 classes precisam de mais de um tópico para cobrir 80% dos artigos?').show()
display(spread.pivot(index=['Classe','Artigos da classe'],columns=['Metodo','K'],values='Topicos para 80%'))

,Metodo,K,Artigos comuns no diagnostico,Topicos usados,Classes espalhadas em 2+ topicos (80%),Mediana de topicos para 80% da classe
0,LLDA + LDA,10,1986,10,54,6.0
1,LLDA + LDA,54,1986,54,54,18.5
2,LDA direto,10,1986,10,53,4.0
3,LDA direto,54,1986,54,54,19.0


Gráfico interativo: comparação dos métodos

Metodo                               LLDA + LDA     LDA direto    
K                                            10  54         10  54
Classe             Artigos da classe                              
cmp-lg             32                         6  16          2  16
cond-mat.dis-nn    19                         5  11          4  13
cond-mat.stat-mech 33                         7  16          5  20
cs.AI              210                        8  29          5  29
cs.CC              101                        7  24          4  24
cs.CE              36                         6  18          4  16
cs.CG              38                         7  15          4  19
cs.CL              97                         7  23          2  25
cs.CR              96                         7  25          6  26
cs.CV              95                         6  23          2  27
cs.CY              60                         6  21          3  16
cs.DB              49                         6  19          5  19
cs.DC              74                         7  26          4  24
cs.DL              31                         6  12          2  16
cs.DM              136                        6  24          5  22
cs.DS              161                        7  27          5  23
cs.FL              37                         5  15          4  20
cs.GT              64                         7  20          6  20
cs.HC              28                         7  15          2  14
cs.IR              77                         7  21          3  25
cs.IT              677                        7  33          4  32
cs.LG              293                        8  31          4  29
cs.LO              129                        6  25          4  26
cs.MA              39                         7  20          5  16
cs.MM              27                         6  18          4  16
cs.MS              12                         6   7          2  10
cs.NA              39                         6  13          4  18
cs.NE              80                         7  20          3  22
cs.NI              106                        7  20          4  25
cs.PF              29                         7  16          3  15
cs.PL              58                         7  23          2  20
cs.RO              34                         7  18          3  19
cs.SC              19                         6  12          4  13
cs.SE              48                         6  20          2  23
cs.SI              133                        6  20          4  24
cs.SY              72                         7  21          4  18
math.CO            119                        6  26          5  21
math.IT            677                        7  33          4  32
math.LO            25                         5  10          4  12
math.NA            37                         6  18          5  15
math.NT            8                          4   7          3   6
math.OC            102                        7  25          4  22
math.PR            35                         7  18          5  17
math.ST            36                         7  16          5  17
nlin.AO            14                         5  11          3  12
physics.data-an    16                         4   8          4   9
physics.soc-ph     113                        5  19          3  23
q-bio.NC           13                         4  10          1   9
q-bio.QM           11                         5   7          3   7
quant-ph           61                         6  16          5  20
stat.AP            21                         5  12          3  11
stat.ME            13                         5  11          5  10
stat.ML            184                        8  28          3  28
stat.TH            36                         7  16          5  17

## 4. Visualizar a relação tópico × classe

Cada coluna é uma classe original. A cor mostra **a porcentagem dos artigos dessa classe** que recebeu cada tópico como dominante; cada coluna soma 100%.

Uma coluna distribuída em várias linhas mostra que a classe foi dividida entre tópicos. Colunas com perfis parecidos mostram classes que compartilham grupos. Isso não estabelece automaticamente erro: a taxonomia é multi-rótulo e pode ter códigos equivalentes.

Os IDs dos tópicos são arbitrários e aprendidos separadamente. Não esperamos uma diagonal: o tópico 1 de um ajuste não é a classe 1 nem necessariamente o tópico 1 de outro ajuste. K=54 iguala os tamanhos, mas não define seus significados.

In [5]:
for method, slug in [('LLDA + LDA','hibrido'),('LDA direto','direto')]:
    for k in [10,54]:
        counts = pd.read_csv(OUTPUT_DIR / f'contagem_topico_classe_{slug}_k{k}.csv', index_col='Topico')
        percentages = counts.div(counts.sum(axis=0), axis=1) * 100
        px.imshow(percentages, zmin=0, zmax=100, aspect='auto', color_continuous_scale='Blues',
                  labels={'x':'Classe original','y':'Tópico dominante','color':'% da classe'},
                  title=f'{method} | K={k}: onde estão os artigos de cada classe?', height=650).show()

Gráfico interativo: comparação dos métodos

Gráfico interativo: comparação dos métodos

Gráfico interativo: comparação dos métodos

Gráfico interativo: comparação dos métodos

## 5. Exemplos concretos sem escolher casos para favorecer um método

Mostramos `cs.IT`, `cs.LG` e `cs.AI`: classes frequentes, com nomes distintos. `cs.IT` e `math.IT` não são tratadas aqui como dois assuntos necessariamente diferentes.

A tabela mostra os tópicos necessários para cobrir 80% e o suporte da classe. Para avaliar se os grupos são bons temas, ainda seriam necessárias leitura dos exemplos, coerência/relevância e, idealmente, avaliação humana. Nenhuma dessas conclusões pode ser substituída por igualdade entre K e 54.

In [6]:
display(spread[spread.Classe.isin(['cs.IT','cs.LG','cs.AI'])])
for (method,k), group in topics.groupby(['Metodo','K'],sort=False):
    chosen = group.nlargest(1,'Artigos').iloc[0]
    doc_ids = assignments.loc[(assignments.Metodo==method) & (assignments.K==k) &
                              (assignments['Topico dominante']==chosen.Topico) & assignments['Incluido no diagnostico'],'doc_id'].head(3)
    print(f'\n{method}, K={k}, tópico {chosen.Topico}: {chosen.Palavras}')
    display(entries[entries.doc_id.isin(doc_ids)][['doc_id','topico_llda']])


LLDA + LDA, K=10, tópico 7: network, social, dynamics, sparse, coding, social network, optimization, network coding

LLDA + LDA, K=54, tópico 8: optimization, network, network optimization, cognitive, radio, communication, cognitive radio, tree

LDA direto, K=10, tópico 3: data, learning, based, paper, results, time, network, performance

LDA direto, K=54, tópico 29: channel, power, user, multiple, interference, transmission, optimal, channels


,Metodo,K,Classe,Artigos da classe,Topicos para 80%
3,LLDA + LDA,10,cs.AI,210,8
20,LLDA + LDA,10,cs.IT,677,7
21,LLDA + LDA,10,cs.LG,293,8
57,LLDA + LDA,54,cs.AI,210,29
74,LLDA + LDA,54,cs.IT,677,33
75,LLDA + LDA,54,cs.LG,293,31
111,LDA direto,10,cs.AI,210,5
128,LDA direto,10,cs.IT,677,4
129,LDA direto,10,cs.LG,293,4
165,LDA direto,54,cs.AI,210,29


,doc_id,topico_llda
2,209,network seed identification measures
4,285,Sparse Matrix Storage Format Adaptation
20,5,Compressive Sensing Sparse Recovery


,doc_id,topico_llda
33,18,Ultra Dense Small Cell Network Optimization
43,28,Cellular Networks Topology Optimization
63,48,influence cascade propagation networks


,doc_id,topico_llda
1,178,Scientific Research Collaboration Dynamics
8,1116,Technical Extractive Summarization
10,1385,input sensitive profiling technique


,doc_id,topico_llda
21,6,Channel Capacity Region
41,26,Spectral modulation techniques
42,27,Resource Allocation Fairness Optimization


## 6. Por que seguir para classificação de tópicos?

O ensaio descreve a estrutura encontrada; não demonstra que a modelagem seja inadequada ou que a classificação tenha maior qualidade. São tarefas diferentes.

Na modelagem, precisamos interpretar e relacionar grupos descobertos à taxonomia, decidir como lidar com subdivisões, classes compartilhadas e múltiplos assuntos por artigo. A avaliação com classes exige escolhas adicionais de correspondência.

Na classificação, definimos o catálogo e seu significado antes da inferência. Cada score já aponta para uma classe conhecida: podemos avaliar diretamente ranking e atribuições multi-rótulo. **Essa clareza de objetivo e avaliação motiva a mudança de tarefa**, mantendo a modelagem como uma linha possível para descoberta de temas.

Não usamos este ensaio para estabelecer vencedor entre LLDA+LDA e LDA direto. Os notebooks antigos permanecem como histórico; esta é a comparação principal simplificada.

## Resultado deste ensaio

Em K=10, 54 classes no híbrido e 53 no LDA direto precisam de mais de um tópico dominante para reunir 80% dos seus artigos. Em K=54, isso ocorre nas 54 classes em ambos os pipelines; a mediana é 18,5 tópicos no híbrido e 19 no direto.

Esses números descrevem a dispersão no conjunto comum de 1.986 artigos. Não medem erro nem definem um vencedor. Eles ilustram que igualar K ao número de classes não cria a correspondência necessária para uma avaliação direta. A classificação explicita esse alvo antes da inferência.